# Test the included spectral-function models

This notebook is a minimal user-facing check for the model files included in this repository. It uses only files shipped with the repository:

- `models/feature_schema.json`
- `models/all_spectral_bins/model_1200.h5`
- `models/all_spectral_bins/model_1600.h5`
- `models/all_spectral_bins/model_2000.h5`
- `examples/model_ready_all_spectral_2000_test_sample.csv`

The example CSV is a small, balanced sample from the standardized 2000 GeV test set. It is included so a user can verify that the model files load, the feature order is clear, and the scoring workflow works end to end.


## Important preprocessing note

The `.h5` models were trained on standardized features. In the training workflow, a `StandardScaler` was fit on the training split and then applied to validation/test data before inference.

The included example CSV is already model-ready. For new event samples, users should provide a table whose columns match `feature_schema.json` and whose values have been standardized using the same convention as the training inputs.


In [ ]:
from pathlib import Path
import json

import pandas as pd
import tensorflow as tf
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score

MODELS_DIR = Path.cwd()

# If you run this notebook from a different working directory, uncomment and edit:
# MODELS_DIR = Path("/path/to/ML_SpectralF_FullyHadronic/models")

REPO_DIR = MODELS_DIR.parent
SCHEMA_PATH = MODELS_DIR / "feature_schema.json"
MODEL_DIR = MODELS_DIR / "all_spectral_bins"
EXAMPLE_CSV = REPO_DIR / "examples" / "model_ready_all_spectral_2000_test_sample.csv"
MASSES = [1200, 1600, 2000]

with SCHEMA_PATH.open("r", encoding="utf-8") as handle:
    schema = json.load(handle)

FEATURES = schema["default_model_features"]
MODEL_PATHS = {mass: MODEL_DIR / f"model_{mass}.h5" for mass in MASSES}

print(f"Models directory: {MODELS_DIR}")
print(f"Example CSV: {EXAMPLE_CSV}")
print(f"Number of model inputs: {len(FEATURES)}")
print(FEATURES)


## Check that all required files are present

In [ ]:
missing = []
if not SCHEMA_PATH.exists():
    missing.append(SCHEMA_PATH)
if not EXAMPLE_CSV.exists():
    missing.append(EXAMPLE_CSV)
for path in MODEL_PATHS.values():
    if not path.exists():
        missing.append(path)

if missing:
    raise FileNotFoundError("Missing required files:\n" + "\n".join(str(path) for path in missing))

for mass, path in MODEL_PATHS.items():
    print(f"{mass}: {path.relative_to(REPO_DIR)}")
print(f"example: {EXAMPLE_CSV.relative_to(REPO_DIR)}")


## End-to-end test with the included example CSV

The example CSV has 20 model-ready events for the 2000 GeV all-spectral-bin setup: 10 background-like rows and 10 signal-like rows. The feature values are already standardized, so they can be fed directly to the `.h5` models.


In [ ]:
example_inputs = pd.read_csv(EXAMPLE_CSV)

missing_columns = [name for name in ["Label"] + FEATURES if name not in example_inputs.columns]
if missing_columns:
    raise ValueError(f"Example CSV is missing required columns: {missing_columns}")

print(example_inputs["Label"].value_counts().sort_index())
example_inputs.head()


In [ ]:
def predict_with_all_models(input_table):
    missing_columns = [name for name in FEATURES if name not in input_table.columns]
    if missing_columns:
        raise ValueError(f"Input table is missing required columns: {missing_columns}")

    x = input_table[FEATURES].to_numpy(dtype="float32")
    scores = pd.DataFrame(index=input_table.index)

    for mass, model_path in MODEL_PATHS.items():
        model = tf.keras.models.load_model(model_path)
        scores[f"score_model_{mass}"] = model.predict(x, verbose=0).ravel()

    return scores


example_scores = predict_with_all_models(example_inputs)
scored_example = pd.concat([example_inputs[["Label"]].reset_index(drop=True), example_scores.reset_index(drop=True)], axis=1)
scored_example


In [ ]:
# The included example is from the 2000 GeV test set, so this quick metric check uses model_2000.
threshold = 0.90
y_true = scored_example["Label"].astype(int)
y_pred = (scored_example["score_model_2000"] >= threshold).astype(int)

print("Accuracy:", accuracy_score(y_true, y_pred))
print("Precision:", precision_score(y_true, y_pred, zero_division=0))
print("Recall:", recall_score(y_true, y_pred, zero_division=0))
print("Confusion matrix [[TN, FP], [FN, TP]]:")
print(confusion_matrix(y_true, y_pred, labels=[0, 1]))


## Test your own model-ready feature table

To test new events, create a CSV with the same columns as `FEATURES`, already standardized using the training preprocessing convention. Then set `CUSTOM_INPUT_CSV` to that file path and run the next cell.

The expected columns are printed below for convenience.


In [ ]:
print("Expected columns:")
for name in FEATURES:
    print(name)

In [ ]:
CUSTOM_INPUT_CSV = None
# Example:
# CUSTOM_INPUT_CSV = "my_standardized_events.csv"

if CUSTOM_INPUT_CSV is None:
    print("Set CUSTOM_INPUT_CSV to a standardized feature CSV to run this section.")
else:
    user_inputs = pd.read_csv(CUSTOM_INPUT_CSV)
    user_scores = predict_with_all_models(user_inputs)
    output_path = Path(CUSTOM_INPUT_CSV).with_name(Path(CUSTOM_INPUT_CSV).stem + "_model_scores.csv")
    pd.concat([user_inputs.reset_index(drop=True), user_scores.reset_index(drop=True)], axis=1).to_csv(output_path, index=False)
    print(f"Saved scores to {output_path}")
    display(user_scores.head())